# Batch Correction for Flow Cytometry

Multi-batch / multi-day flow experiments accumulate technical variation (laser drift, reagent lots, instrument settings) that swamps biological signal. R has the strongest tooling for this — three Saeys-lab packages cover the full workflow:

| Step | Package | Role |
|------|---------|------|
| QC   | [**PeacoQC**](https://github.com/saeyslab/PeacoQC) | Removes margin events and acquisition anomalies (clogs, speed changes) |
| Correction (no anchors) | [**cyCombine**](https://github.com/biosurf/cyCombine) | SOM clustering + per-cluster ComBat. Works without shared reference samples. *Pedersen et al., Nat Commun 2022.* |
| Correction (with anchors) | [**CytoNorm**](https://github.com/saeyslab/CytoNorm) | Quantile normalization trained on aliquoted control samples present in every batch. *Van Gassen et al., Cytometry A 2020.* |

**Decision tree:**
- Same reference control aliquoted into every batch? → use **CytoNorm**.
- No shared anchor samples? → use **cyCombine** with `covar = "condition"` to preserve biology.

This notebook runs the full pipeline: QC → batch-effect detection → correction → diagnostics → save.

## 1. Setup

Install the three batch-correction packages once (they're not on CRAN). cyCombine pulls `sva` from Bioconductor for the ComBat backend.

```r
BiocManager::install(c("sva", "flowCore", "flowWorkspace", "FlowSOM", "ggcyto", "ComplexHeatmap"))
remotes::install_github("saeyslab/PeacoQC")
remotes::install_github("saeyslab/CytoNorm")
remotes::install_github("biosurf/cyCombine")
```

In [ ]:
library(flowCore)
library(flowWorkspace)
library(ggcyto)
library(FlowSOM)
library(PeacoQC)
library(CytoNorm)
library(cyCombine)
library(ggplot2)
library(dplyr)
library(tidyr)
library(uwot)
library(viridis)
library(patchwork)

set.seed(42)

## 2. Load batch and define metadata

Each FCS file needs to be tagged with at minimum a **Batch** ID (the technical grouping you want to correct away) and a **Sample** ID. If you also have biological condition labels (treatment, timepoint, donor), include them — `cyCombine` can use them as `covar` to preserve real signal.

**Template note:** the demo only ships one `sample.fcs`, so this cell synthesizes two pseudo-batches by splitting that file in half. Replace the metadata block with your real annotation when you have multiple files.

In [ ]:
fcs_dir   <- "../data/batch/"
fcs_files <- list.files(fcs_dir, pattern = "\\.fcs$", full.names = TRUE)

# -- Build metadata --------------------------------------------------------
# Template (real workflow): one row per FCS file.
# metadata <- data.frame(
#     FCS_name = basename(fcs_files),
#     Sample   = sub("\\.fcs$", "", basename(fcs_files)),
#     Batch    = c("B1", "B1", "B2", "B2", ...),       # your batch labels
#     condition = c("ctrl", "trt", "ctrl", "trt", ...)  # optional, for covar
# )

# Helper: fill missing parameter descriptions ONLY for fluorescence channels.
# cyCombine uses `desc` as the marker label and drops channels whose desc is
# NA — so any unlabelled fluorescence channel needs a fallback. Leave
# scatter/Time desc as NA so cyCombine treats them as non-markers.
fill_desc <- function(ff) {
    p <- pData(parameters(ff))
    miss <- is.na(p$desc) & !grepl("^(FSC|SSC|Time)", p$name)
    if (any(miss)) {
        p$desc[miss] <- p$name[miss]
        pData(parameters(ff)) <- p
    }
    ff
}

# -- Demo: split single FCS into two pseudo-batches so the pipeline runs --
if (length(fcs_files) == 1) {
    ff <- read.FCS(fcs_files, transformation = FALSE, truncate_max_range = FALSE)
    ff <- fill_desc(ff)
    n  <- nrow(ff)
    half <- floor(n / 2)
    ff_a <- ff[1:half, ]
    ff_b <- ff[(half + 1):n, ]
    # Small multiplicative shift on the *positive* side of fluorescence channels.
    # Keeps the negative tail intact so estimateLogicle remains stable.
    e <- exprs(ff_b)
    fl_cols <- grep("^(FSC|SSC|Time)", colnames(e), invert = TRUE)
    for (j in fl_cols) {
        pos <- e[, j] > 0
        e[pos, j] <- e[pos, j] * 1.15    # ~15% gain on positives = visible batch effect
    }
    exprs(ff_b) <- e
    demo_dir <- tempfile("batchdemo_"); dir.create(demo_dir)
    write.FCS(ff_a, file.path(demo_dir, "S01_B1.fcs"))
    write.FCS(ff_b, file.path(demo_dir, "S02_B2.fcs"))
    fcs_dir   <- demo_dir
    fcs_files <- list.files(fcs_dir, pattern = "\\.fcs$", full.names = TRUE)
    metadata  <- data.frame(
        FCS_name = basename(fcs_files),
        Sample   = c("S01", "S02"),
        Batch    = c("B1",  "B2"),
        condition = c("ctrl", "ctrl"),
        stringsAsFactors = FALSE
    )
    cat("Demo mode: created 2 pseudo-batches in", demo_dir, "\n")
} else {
    # -- Replace this with your real metadata --
    metadata <- data.frame(
        FCS_name = basename(fcs_files),
        Sample   = sub("\\.fcs$", "", basename(fcs_files)),
        Batch    = sub(".*_(B[0-9]+)\\.fcs$", "\\1", basename(fcs_files)),
        condition = "ctrl",
        stringsAsFactors = FALSE
    )
}

metadata

fs <- read.flowSet(fcs_files, transformation = FALSE, truncate_max_range = FALSE)
fs <- fsApply(fs, fill_desc)
pData(fs)$name  <- metadata$Sample
pData(fs)$Batch <- metadata$Batch

all_channels   <- colnames(fs)
fluor_channels <- grep("^(FSC|SSC|Time)", all_channels, value = TRUE, invert = TRUE)
cat("Fluorescence channels:", paste(fluor_channels, collapse = ", "), "\n")


## 3. Quality control with PeacoQC

Before any batch correction, drop **margin events** (saturated at detector min/max) and **acquisition anomalies** (clogs, flow rate changes, instability over time). PeacoQC scans every channel for peaks and removes time-windows where the peak structure breaks.

Order matters: `RemoveMargins` → `compensate` → `transform` → `PeacoQC`.

In [ ]:
qc_dir <- file.path(tempdir(), "peacoqc_out"); dir.create(qc_dir, showWarnings = FALSE)

get_spill <- function(ff) {
    spill <- keyword(ff)$SPILL
    if (is.null(spill)) spill <- keyword(ff)$`$SPILLOVER`
    spill
}

# Compensate every sample first (uses each file's own spillover)
fs_comp <- fsApply(fs, function(ff) {
    sp <- get_spill(ff)
    if (!is.null(sp)) compensate(ff, sp) else ff
})

# Estimate ONE logicle transform from the first compensated sample, then apply
# the same transform to every sample. This keeps cross-sample comparisons valid
# and avoids per-sample logicle failures on edge-case distributions.
lgcl <- estimateLogicle(fs_comp[[1]], channels = fluor_channels)
fs_trans <- transform(fs_comp, lgcl)

# Per-sample: RemoveMargins (on transformed data) → PeacoQC
qc_frames <- lapply(seq_along(fs_trans), function(i) {
    ff <- fs_trans[[i]]
    n_in <- nrow(ff)

    ff <- RemoveMargins(ff = ff, channels = fluor_channels, output = "frame")

    res <- PeacoQC(
        ff             = ff,
        channels       = fluor_channels,
        determine_good_cells = "all",
        save_fcs       = FALSE,
        plot           = FALSE,
        output_directory = qc_dir
    )
    list(ff = res$FinalFF, n_before = n_in, n_after = nrow(res$FinalFF))
})

fs_qc <- as(lapply(qc_frames, `[[`, "ff"), "flowSet")
sampleNames(fs_qc) <- sampleNames(fs)
pData(fs_qc) <- pData(fs)

qc_summary <- data.frame(
    sample   = sampleNames(fs),
    batch    = pData(fs)$Batch,
    n_before = sapply(qc_frames, `[[`, "n_before"),
    n_after  = sapply(qc_frames, `[[`, "n_after")
)
qc_summary$pct_retained <- round(100 * qc_summary$n_after / qc_summary$n_before, 1)
qc_summary


## 4. Detect batch effects

Two visual diagnostics decide whether correction is needed:

1. **Marker density overlays per batch** — if curves diverge, you have a batch effect.
2. **UMAP colored by batch** — biological structure should not separate by batch.

In [ ]:
# Stack QC'd data across samples
df_qc <- bind_rows(lapply(seq_along(fs_qc), function(i) {
    d <- as.data.frame(exprs(fs_qc[[i]]))
    d$Sample <- sampleNames(fs_qc)[i]
    d$Batch  <- pData(fs_qc)$Batch[i]
    d
}))

# 1) Density overlays per batch — the canonical batch-effect plot
df_long <- df_qc %>%
    select(all_of(fluor_channels), Batch) %>%
    pivot_longer(-Batch, names_to = "marker", values_to = "intensity")

g_density <- ggplot(df_long, aes(x = intensity, color = Batch)) +
    geom_density(linewidth = 0.6) +
    facet_wrap(~marker, scales = "free") +
    theme_bw() +
    labs(title = "Per-marker density by batch (BEFORE correction)",
         subtitle = "Divergent curves = batch effect")
g_density

In [ ]:
# 2) UMAP colored by batch
set.seed(42)
n_sub <- min(nrow(df_qc), 20000)
idx   <- sample(nrow(df_qc), n_sub)
mat   <- scale(as.matrix(df_qc[idx, fluor_channels]))
umap_before <- uwot::umap(mat, n_neighbors = 15, min_dist = 0.2)

umap_df <- data.frame(
    UMAP1 = umap_before[, 1],
    UMAP2 = umap_before[, 2],
    Batch = df_qc$Batch[idx],
    Sample = df_qc$Sample[idx]
)

ggplot(umap_df, aes(UMAP1, UMAP2, color = Batch)) +
    geom_point(alpha = 0.35, size = 0.3) +
    theme_bw() +
    labs(title = "UMAP — BEFORE correction",
         subtitle = "Batches separating into distinct regions = batch effect to correct")

## 5. Correct with cyCombine (general-purpose, no anchor required)

cyCombine's workflow:
1. **`prepare_data`** — pools FCS files, applies `arcsinh(x / cofactor)` transform, attaches metadata.
2. **`batch_correct`** — builds a self-organizing map across all cells, then runs ComBat *within each SOM node* (preserves biological structure while removing batch shifts).

**Key parameters:**
- `cofactor` — 5 for CyTOF, 150–500 for conventional fluorescence flow. Tune so populations look well-separated.
- `norm_method` — `"scale"` (default, gentler) or `"rank"` (heavier batch effects).
- `covar` — name of a biological condition column to *preserve* during ComBat.
- `xdim` × `ydim` — SOM grid size (8×8 = 64 nodes is a good default).

In [ ]:
# cyCombine reads FCS files itself (it re-transforms with arcsinh), so we point
# it at the original (or margin-cleaned) files, NOT the logicle-transformed ones.

# Build an explicit panel: maps the FCS detector channel (`fcs_colname`) to a
# clean marker name (`antigen`). Passing this avoids cyCombine producing
# NA-named columns for unlabeled channels — which breaks dplyr's all_of() in
# transform_asinh.
ff_first   <- read.FCS(fcs_files[1], transformation = FALSE, truncate_max_range = FALSE)
ff_first   <- fill_desc(ff_first)
param_meta <- pData(parameters(ff_first))
fluor_idx  <- !grepl("^(FSC|SSC|Time)", param_meta$name)
panel <- data.frame(
    fcs_colname = param_meta$name[fluor_idx],
    antigen     = gsub("-", "", param_meta$desc[fluor_idx]),  # cyCombine strips hyphens
    stringsAsFactors = FALSE
)
markers <- panel$antigen
cat("cyCombine panel:\n"); print(panel)

# derand = FALSE: the default (TRUE) only makes sense for CyTOF data with
# integer counts; on float fluorescence values it fails marker lookup.
uncorrected <- prepare_data(
    data_dir     = fcs_dir,
    metadata     = metadata,
    filename_col = "FCS_name",
    batch_ids    = "Batch",
    sample_ids   = "Sample",
    condition    = "condition",
    panel        = panel,
    panel_channel = "fcs_colname",
    panel_antigen = "antigen",
    markers      = markers,
    down_sample  = FALSE,
    cofactor     = 150,
    derand       = FALSE
)
cat("Uncorrected:", nrow(uncorrected), "cells across",
    length(unique(uncorrected$batch)), "batches\n")
head(uncorrected)


In [ ]:
corrected <- uncorrected %>%
    batch_correct(
        markers     = markers,
        norm_method = "scale",
        xdim        = 8,
        ydim        = 8,
        covar       = "condition",
        seed        = 42
    )
cat("Corrected:", nrow(corrected), "cells\n")
head(corrected)

## 6. Diagnostics — before vs. after

Three checks:
- **Density overlay** — batch curves should converge.
- **UMAP** — batches should overlap, biological structure should remain.
- **EMD reduction** — quantitative metric (>0 = correction helped). See [cyCombine benchmarking vignette](https://biosurf.org/cyCombine_benchmarking.html).

In [ ]:
# cyCombine's built-in density comparison
plot_density(uncorrected, corrected, markers = markers, ncol = 2)

In [ ]:
# UMAP before/after — colored by batch
set.seed(42)
n_plot <- min(nrow(uncorrected), 20000)
idx_u  <- sample(nrow(uncorrected), n_plot)
idx_c  <- sample(nrow(corrected),   n_plot)

p_before <- plot_dimred(uncorrected[idx_u, ], type = "umap", name = "Uncorrected") +
    ggtitle("BEFORE — UMAP by batch")
p_after  <- plot_dimred(corrected[idx_c, ],   type = "umap", name = "Corrected")   +
    ggtitle("AFTER — UMAP by batch")

p_before + p_after

In [ ]:
# EMD (Earth Mover's Distance) reduction is cyCombine's headline metric:
#   reduction > 0 → batch effect was reduced.
# evaluate_emd needs the same SOM cluster column on both data frames.
# batch_correct() preserves row order, so we copy the SOM labels across.
tryCatch({
    uncorrected_eval <- uncorrected
    uncorrected_eval$som <- corrected$som

    emd <- evaluate_emd(uncorrected_eval, corrected,
                        cell_col = "som", markers = markers, binSize = 0.1)
    cat("Mean EMD reduction:", round(emd$reduction, 3),
        "  (positive = batch effect reduced)\n")
    print(head(emd$emd))
}, error = function(e) {
    message("evaluate_emd failed (often happens with very small demo data): ",
            conditionMessage(e))
})


## 7. Alternative — CytoNorm (when you have anchor controls)

If every batch contains the same aliquoted reference sample (e.g., a frozen PBMC pool), CytoNorm typically outperforms ComBat-based methods. It trains a per-cluster quantile spline that maps each batch's reference back to a common goal distribution, then applies the same transform to all other samples in the batch.

**Required metadata:** add a `Type` column with values `Train` (anchor controls) and `Validation` (samples to correct).

Skip this section if you don't have anchor controls — `cyCombine` from section 5 is your correction.

In [ ]:
# Skip if no anchor controls
has_anchors <- "Type" %in% colnames(metadata) && any(metadata$Type == "Train")

if (has_anchors) {
    train_data <- metadata %>% filter(Type == "Train")
    val_data   <- metadata %>% filter(Type == "Validation")

    transformList <- transformList(fluor_channels, logicleTransform())
    transformList.reverse <- transformList(fluor_channels, inverseLogicleTransform(trans = logicleTransform()))

    # Train
    model <- CytoNorm.train(
        files          = file.path(fcs_dir, train_data$FCS_name),
        labels         = train_data$Batch,
        channels       = fluor_channels,
        transformList  = transformList,
        FlowSOM.params = list(nCells = 6000, xdim = 5, ydim = 5, nClus = 10, scale = FALSE),
        normMethod.train = QuantileNorm.train,
        normParams     = list(nQ = 101, goal = "mean"),
        seed           = 42,
        verbose        = TRUE
    )

    # Apply to validation samples — writes corrected FCS files to outputDir
    out_norm <- file.path("../data/results", "04_cytonorm_normalized")
    dir.create(out_norm, showWarnings = FALSE, recursive = TRUE)
    CytoNorm.normalize(
        model                   = model,
        files                   = file.path(fcs_dir, val_data$FCS_name),
        labels                  = val_data$Batch,
        transformList           = transformList,
        transformList.reverse   = transformList.reverse,
        normMethod.normalize    = QuantileNorm.normalize,
        outputDir               = out_norm,
        prefix                  = "Norm_",
        verbose                 = TRUE
    )
    cat("CytoNorm output:", out_norm, "\n")
} else {
    message("No 'Type' column with 'Train' anchors — skipping CytoNorm. Use cyCombine output (section 5).")
}

## 8. Save corrected data

cyCombine returns a `data.frame` keyed on `sample` + `batch`. We round-trip it back to FCS files (one per sample), so downstream tooling can pick up the corrected expression without modification.

In [ ]:
out_dir <- "../data/results/04_corrected_fcs"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# Reverse cyCombine's arcsinh(x/cofactor) so saved FCS values are on the original scale.
cofactor <- 150
for (s in unique(corrected$sample)) {
    cells <- corrected %>% filter(sample == s)
    expr  <- as.matrix(cells[, markers]) * 0
    expr[, markers] <- sinh(as.matrix(cells[, markers])) * cofactor
    ff_out <- flowFrame(expr)
    write.FCS(ff_out, file.path(out_dir, paste0("corrected_", s, ".fcs")))
}

# Also save the long-format data.frame for downstream analysis
write.csv(corrected,   file.path("../data/results", "04_corrected_long.csv"),   row.names = FALSE)
write.csv(uncorrected, file.path("../data/results", "04_uncorrected_long.csv"), row.names = FALSE)
write.csv(qc_summary,  file.path("../data/results", "04_qc_summary.csv"),       row.names = FALSE)
if (exists("emd")) write.csv(emd$emd, file.path("../data/results", "04_emd.csv"))

cat("Saved corrected FCS files to:", out_dir, "\n")
cat("Downstream: feed these into notebook 03 for clustering on corrected data.\n")